In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os
import sys
import json
from pathlib import Path

# Définition et ajout du chemin vers le dépôt AASIST
repo_dir = "/content/drive/MyDrive/aasist"

if os.path.exists(repo_dir):
    os.chdir(repo_dir)
    if repo_dir not in sys.path:
        sys.path.append(repo_dir)
    print(f"Dossier de travail actif : {os.getcwd()}")
else:
    print(f"ERREUR : Le chemin {repo_dir} est introuvable.")
    print("Vérifie que tu as bien créé le 'Raccourci' dans Google Drive sur ce second compte !")

# 3. Imports généraux et vérification du GPU
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchaudio
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Appareil de calcul utilisé : {device}")

if torch.cuda.is_available():
    print(f"Nom du GPU : {torch.cuda.get_device_name(0)}")
    print(f"Mémoire GPU totale : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} Go")

# 4. Import des modules spécifiques d'AASIST
from models.AASIST import Model as AASISTModel
import evaluation as eval_metric

print("Étape 1 : Modules AASIST et GPU prêts ")

Dossier de travail actif : /content/drive/MyDrive/aasist
Appareil de calcul utilisé : cuda
Nom du GPU : Tesla T4
Mémoire GPU totale : 15.64 Go
Étape 1 : Modules AASIST et GPU prêts 


In [ ]:
import os
import kagglehub
from pathlib import Path

# 1. Téléchargement et localisation des données
path = kagglehub.dataset_download("awsaf49/asvpoof-2019-dataset")
base_dir = Path(path)

# 2. Recherche ultra-rapide des fichiers via os.walk (FLAC et TXT)
print("Indexation des fichiers en cours...")
flac_map = {}
train_txt, dev_txt, eval_txt = None, None, None

for root, _, files in os.walk(base_dir):
    # Filtrer pour ne garder que le dossier Logical Access (LA)
    if "LA" in root:
        for f in files:
            full_path = Path(root) / f
            if f.endswith(".flac"):
                flac_map[f[:-5]] = full_path
            elif f.endswith("train.trn.txt"):
                train_txt = full_path
            elif f.endswith("dev.trl.txt"):
                dev_txt = full_path
            elif f.endswith("eval.trl.txt"):
                eval_txt = full_path

# 3. Fonction de parsing des fichiers de protocole
def parse_protocol(protocol_file):
    file_paths = []
    labels = {}
    with open(protocol_file, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                key, label = parts[1], parts[4]
                if key in flac_map:
                    file_paths.append(flac_map[key])
                    labels[key] = 1 if label == "bonafide" else 0
    return file_paths, labels

# 4. Parsing des sets
paths_train, labels_train = parse_protocol(train_txt)
paths_dev, labels_dev = parse_protocol(dev_txt)
paths_eval, labels_eval = parse_protocol(eval_txt)

print(f"--- Étape 2 terminée ---")
print(f"Audios chargés -> Train: {len(paths_train)} | Dev: {len(paths_dev)} | Eval: {len(paths_eval)}")

Using Colab cache for faster access to the 'asvpoof-2019-dataset' dataset.
Indexation des fichiers en cours...


KeyboardInterrupt: 

In [ ]:
# 1. Fonction d'ajustement de la taille audio (fixée à 64 600 échantillons)
def prepare_audio(path):
    waveform, _ = torchaudio.load(str(path))
    audio = waveform.squeeze(0)

    if len(audio) < 64600:
        audio = audio.repeat((64600 // len(audio)) + 1)[:64600]
    else:
        audio = audio[:64600]

    return audio.float()

# 2. Classe PyTorch Dataset
class ASVDataset(Dataset):
    def __init__(self, paths, labels):
        self.paths = paths
        self.labels = labels

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        key = path.stem
        audio = prepare_audio(path)
        label = torch.tensor(self.labels[key], dtype=torch.long)
        return audio, label, key

print("Étape 3 terminée : Dataset et préparation audio")

NameError: name 'Dataset' is not defined

In [ ]:
# 1. Chargement du fichier de configuration AASIST
config_path = f"{repo_dir}/config/AASIST.conf"

with open(config_path, "r") as f:
    config = json.load(f)

# 2. Initialisation du modèle et transfert sur le device (GPU)
model = AASISTModel(config["model_config"]).to(device)

print("--- Étape 4 terminée : Modèle AASIST initialisé avec succès ! ---")

--- Étape 4 terminée : Modèle AASIST initialisé avec succès ! ---


In [ ]:
# 1. Fonction d'entraînement pour une époque
def train_epoch(model, train_loader, optimizer, criterion, device):
    model.train()
    total_loss, right_predicts, total_samples = 0.0, 0, 0

    for x_batch, y_batch, _ in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        _, outputs = model(x_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        batch_size = y_batch.size(0)
        total_loss += loss.item() * batch_size
        predicts = torch.argmax(outputs, dim=1)
        right_predicts += torch.sum(predicts == y_batch).item()
        total_samples += batch_size

    return right_predicts / total_samples, total_loss / total_samples

# 2. Fonction de prédiction sur le set de validation/évaluation
def eval_epoch(model, dev_loader, device):
    model.eval()
    tracker = {}

    with torch.no_grad():
        for x_batch, _, keys in dev_loader:
            x_batch = x_batch.to(device)
            _, outputs = model(x_batch)
            bonafide_scores = outputs[:, 1].cpu().numpy()
            for key, score in zip(keys, bonafide_scores):
                tracker[key] = score

    return tracker

# 3. Fonction de calcul de l'EER (Equal Error Rate)
def compute_eer(tracker, protocol_file):
    target_scores, nontarget_scores = [], []
    with open(protocol_file, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                key, label = parts[1], parts[4]
                if key in tracker:
                    if label == "bonafide":
                        target_scores.append(tracker[key])
                    else:
                        nontarget_scores.append(tracker[key])

    eer, _ = eval_metric.compute_eer(np.array(target_scores), np.array(nontarget_scores))
    return eer * 100

print("--- Étape 5 terminée : Fonctions d'entraînement et calcul EER prêtes ! ---")

--- Étape 5 terminée : Fonctions d'entraînement et calcul EER prêtes ! ---


In [ ]:
# 1. Configuration des DataLoaders & Hyperparamètres
BATCH_SIZE = 32
train_loader = DataLoader(ASVDataset(paths_train, labels_train), batch_size=BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(ASVDataset(paths_dev, labels_dev), batch_size=BATCH_SIZE, shuffle=False)

# 2. Perte (CrossEntropy avec poids) & Optimiseur Adam
weights = torch.FloatTensor([0.1, 0.9]).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=config["optim_config"]["base_lr"],
    weight_decay=config["optim_config"]["weight_decay"]
)

NUM_EPOCHS = 10
BEST_EER = 100.0
SAVE_DIR = "/content/drive/MyDrive/aasist/checkpoints"
os.makedirs(SAVE_DIR, exist_ok=True)
MODEL_SAVE_PATH = os.path.join(SAVE_DIR, "best_aasist_model.pth")

# 3. Reprise automatique depuis le checkpoint de Google Drive
if os.path.exists(MODEL_SAVE_PATH):
    model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
    print("Modèle précédent trouvé sur Drive ! Chargement et évaluation de l'EER initial...")
    initial_tracker = eval_epoch(model, dev_loader, device)
    BEST_EER = compute_eer(initial_tracker, dev_txt)
    print(f"Meilleur EER actuel (référence) : {BEST_EER:.2f}%")

# 4. Boucle principale d'entraînement
print("\n--- Début de l'entraînement sur GPU ---")
for epoch in range(1, NUM_EPOCHS + 1):
    print(f"\nÉPOQUE [{epoch}/{NUM_EPOCHS}]")
    train_acc, train_loss = train_epoch(model, train_loader, optimizer, criterion, device)
    print(f"  [TRAIN] Loss : {train_loss:.4f} | Accuracy : {train_acc * 100:.2f}%")

    tracker = eval_epoch(model, dev_loader, device)
    dev_eer = compute_eer(tracker, dev_txt)
    print(f"  [EVAL] Dev EER : {dev_eer:.2f}%")

    # Sauvegarde si nouveau record
    if dev_eer < BEST_EER:
        print(f"  --> NOUVEAU RECORD ! EER : {BEST_EER:.2f}% -> {dev_eer:.2f}%")
        BEST_EER = dev_eer
        torch.save(model.state_dict(), MODEL_SAVE_PATH)

print("\n--- Entraînement terminé ---")


--- Début de l'entraînement sur GPU ---

ÉPOQUE [1/10]
  [TRAIN] Loss : 0.5280 | Accuracy : 74.01%
  [EVAL] Dev EER : 10.47%
  --> NOUVEAU RECORD ! EER : 100.00% -> 10.47%

ÉPOQUE [2/10]
  [TRAIN] Loss : 0.2011 | Accuracy : 91.90%
  [EVAL] Dev EER : 6.24%
  --> NOUVEAU RECORD ! EER : 10.47% -> 6.24%

ÉPOQUE [3/10]
  [TRAIN] Loss : 0.1093 | Accuracy : 96.08%
  [EVAL] Dev EER : 3.22%
  --> NOUVEAU RECORD ! EER : 6.24% -> 3.22%

ÉPOQUE [4/10]
  [TRAIN] Loss : 0.0820 | Accuracy : 97.29%
  [EVAL] Dev EER : 1.57%
  --> NOUVEAU RECORD ! EER : 3.22% -> 1.57%

ÉPOQUE [5/10]
  [TRAIN] Loss : 0.0568 | Accuracy : 98.18%
  [EVAL] Dev EER : 2.13%

ÉPOQUE [6/10]
  [TRAIN] Loss : 0.0439 | Accuracy : 98.52%
  [EVAL] Dev EER : 2.43%

ÉPOQUE [7/10]
  [TRAIN] Loss : 0.0358 | Accuracy : 98.92%
  [EVAL] Dev EER : 1.45%
  --> NOUVEAU RECORD ! EER : 1.57% -> 1.45%

ÉPOQUE [8/10]
  [TRAIN] Loss : 0.0412 | Accuracy : 98.76%


In [ ]:
# Cellule 7 : Évaluation finale sur le jeu d'évaluation (Eval set)
print("--- Chargement du meilleur modèle pour l'évaluation ---")

# 1. Chargement des poids sauvegardés
if os.path.exists(MODEL_SAVE_PATH):
    model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
    print(f"Modèle chargé avec succès depuis : {MODEL_SAVE_PATH}")
else:
    print(f"Erreur : Aucun fichier trouvé à l'emplacement {MODEL_SAVE_PATH}")

# 2. Création du DataLoader d'évaluation
eval_loader = DataLoader(ASVDataset(paths_eval, labels_eval), batch_size=BATCH_SIZE, shuffle=False)

# 3. Calcul des scores et de l'EER sur Eval
print("Évaluation en cours sur le jeu Eval (71 237 audios)...")
eval_tracker = eval_epoch(model, eval_loader, device)
final_eval_eer = compute_eer(eval_tracker, eval_txt)

print("\n" + "="*40)
print(f" RESULTAT FINAL EVAL EER : {final_eval_eer:.2f}%")
print("="*40)

--- Chargement du meilleur modèle pour l'évaluation ---


NameError: name 'os' is not defined

In [ ]:
import os
import sys
import json
import numpy as np
import torch
import torch.nn as nn
import torchaudio
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import kagglehub

# 1. Configuration des chemins
repo_dir = "/content/drive/MyDrive/aasist"
if repo_dir not in sys.path:
    sys.path.append(repo_dir)

from models.AASIST import Model as AASISTModel
import evaluation as eval_metric

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. Localisation et indexation ultra-rapide des audios
path = kagglehub.dataset_download("awsaf49/asvpoof-2019-dataset")
base_dir = Path(path)

print("Indexation rapide des fichiers pour l'évaluation...")
flac_map = {}
eval_txt = None

for root, _, files in os.walk(base_dir):
    if "LA" in root:
        for f in files:
            full_path = Path(root) / f
            if f.endswith(".flac"):
                flac_map[f[:-5]] = full_path
            elif f.endswith("eval.trl.txt"):
                eval_txt = full_path

# 3. Parsing du protocole d'évaluation
paths_eval, labels_eval = [], {}
with open(eval_txt, "r") as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) >= 5:
            key, label = parts[1], parts[4]
            if key in flac_map:
                paths_eval.append(flac_map[key])
                labels_eval[key] = 1 if label == "bonafide" else 0

# 4. Classes et fonctions d'évaluation
def prepare_audio(path):
    waveform, _ = torchaudio.load(str(path))
    audio = waveform.squeeze(0)
    if len(audio) < 64600:
        audio = audio.repeat((64600 // len(audio)) + 1)[:64600]
    else:
        audio = audio[:64600]
    return audio.float()

class ASVDataset(Dataset):
    def __init__(self, paths, labels):
        self.paths = paths
        self.labels = labels

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        path = self.paths[idx]
        key = path.stem
        audio = prepare_audio(path)
        label = torch.tensor(self.labels[key], dtype=torch.long)
        return audio, label, key

def eval_epoch(model, dev_loader, device):
    model.eval()
    tracker = {}
    with torch.no_grad():
        for x_batch, _, keys in dev_loader:
            x_batch = x_batch.to(device)
            _, outputs = model(x_batch)
            bonafide_scores = outputs[:, 1].cpu().numpy()
            for key, score in zip(keys, bonafide_scores):
                tracker[key] = score
    return tracker

def compute_eer(tracker, protocol_file):
    target_scores, nontarget_scores = [], []
    with open(protocol_file, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) >= 5:
                key, label = parts[1], parts[4]
                if key in tracker:
                    if label == "bonafide":
                        target_scores.append(tracker[key])
                    else:
                        nontarget_scores.append(tracker[key])
    eer, _ = eval_metric.compute_eer(np.array(target_scores), np.array(nontarget_scores))
    return eer * 100

# 5. Chargement du modèle AASIST et du checkpoint
config_path = f"{repo_dir}/config/AASIST.conf"
with open(config_path, "r") as f:
    config = json.load(f)

model = AASISTModel(config["model_config"]).to(device)
MODEL_SAVE_PATH = f"{repo_dir}/checkpoints/best_aasist_model.pth"

if os.path.exists(MODEL_SAVE_PATH):
    model.load_state_dict(torch.load(MODEL_SAVE_PATH, map_location=device))
    print(f"Modèle répertorié chargé depuis : {MODEL_SAVE_PATH}")

    # 6. Exécution de l'évaluation
    eval_loader = DataLoader(ASVDataset(paths_eval, labels_eval), batch_size=32, shuffle=False)
    print("Évaluation en cours sur le jeu Eval (71 237 audios)...")
    eval_tracker = eval_epoch(model, eval_loader, device)
    final_eval_eer = compute_eer(eval_tracker, eval_txt)

    print("\n" + "="*40)
    print(f" RESULTAT FINAL EVAL EER : {final_eval_eer:.2f}%")
    print("="*40)
else:
    print(f"ERREUR : Fichier modèle introuvable sur {MODEL_SAVE_PATH}")

100%|██████████| 23.6G/23.6G [03:19<00:00, 127MB/s] 

Extracting files...


Indexation rapide des fichiers pour l'évaluation...
Modèle répertorié chargé depuis : /content/drive/MyDrive/aasist/checkpoints/best_aasist_model.pth
Évaluation en cours sur le jeu Eval (71 237 audios)...
